Repo URL: https://github.com/<redacted>/assignment-4-equation-of-a-slime-<redacted>
Commit ID: 77766f4f2b810b99256f18bf1ec2ee23011e8712

In [134]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression 
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
import random

## Part 1. Loading the dataset

In [135]:
# Using pandas load the dataset (load remotely, not locally)
df = pd.read_csv('/workspaces/assignment-4-equation-of-a-slime-<redacted>/science_data_large.csv')

# Output the first 15 rows of the data
print(df.head(15))

# Display a summary of the table information (number of datapoints, etc.)
print(df.describe())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   226.750000  1.298267e+05
50%        500.500

## Part 2. Splitting the dataset

In [136]:
# Take the pandas dataset and split it into our features (X) and label (y)
# FEATURE IS INPUT, LABEL IS OUTPUT
# FEATURE IS temp and mols, label is Size nm^3

X = df.drop("Size nm^3", axis = 1)
y = df.drop(["Temperature °C", "Mols KCL"], axis = 1)
print(X.head(15))
# print(y.head(15))

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.10, random_state = 42)

    Temperature °C  Mols KCL
0              469       647
1              403       694
2              302       975
3              779       916
4              901        18
5              545       637
6              660       519
7              143       869
8               89       461
9              294       776
10             991       117
11             307       781
12             206        70
13             437       599
14             566        75


## Part 3. Perform a Linear Regression

In [137]:
# Use sklearn to train a model on the training set
regr = LinearRegression() 
  
regr.fit(X_train, y_train) 
print(round(regr.score(X_test, y_test), 5)) # score based on test data

# Create a sample datapoint and predict the output of that sample with the trained model
sample_datapoint = pd.DataFrame(data = {"Temperature °C" : [469], "Mols KCL" : [647]}) #[469, 647]
print(sample_datapoint)

sample_pred = regr.predict(sample_datapoint)
print(sample_pred)

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
# see below for markdown
# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
# Get the coefficients
coefficients = regr.coef_
print(coefficients)

# Get the intercept
intercept = regr.intercept_
print(intercept)

# equation is below

0.85525
   Temperature °C  Mols KCL
0             469       647
[[664984.89630433]]
[[ 866.14641337 1032.69506649]]
[-409391.47958341]


Explanation of the Model Score

The score produced by the `regr.score(X_test, y_test)` function is the coefficient of determination $R^2$, which indicates how well the model fits the test data. The model is fit based off of our train data, then we score it against the test data.

- A score of **1** means that the model perfectly predicts the target variable, explaining all of the variance in the data.
- A score of **0** means the model explains none of the variance, performing no better than predicting the mean of the target variable.
- A negative score suggests the model performs worse than a horizontal line predicting the mean.

Thus, the \(R^2\) score provides a measure of the model's goodness-of-fit: the closer the score is to **1**, the better the model is at making predictions.


<!-- Sample equation: $E = mc^2$ -->
Equation for regression: $h(x) = -409391.47958 + 866.14641 * Temperature °C + 1032.69507 * Mols KCL$

## Part 4. Use Cross Validation

In [138]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
scores = cross_val_score(regr, X, y, cv=5)
print(scores)
print(round(sum(scores)/len(scores), 5))
print(round(min(scores), 5), round(max(scores), 5))

# Report on their finding and their significance

[0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
0.85682
0.83919 0.87203


#### **Report on their finding and their significance.**

Findings:
These scores represent the performance of your linear regression model on different subsets (folds) of the data. Each number is an $R^2$ score (coefficient of determination) that measures how well the model explains the variance in the target variable for each fold.

**Consistency Across Folds:** The scores range from approximately 0.83919 to 0.87203, which indicates a relatively stable performance of the model across different folds of the data. The fact that the scores are generally close to each other with a small range suggests that the model is generalizing well, and there is no major overfitting or underfitting issue. However, we know that the more folds we use (5 here), there will likely be a greater range.

**Good Model Performance:** All scores are above 0.83, which indicates that the model explains more than 83% of the variance in the target variable. This is a good performance, particularly for regression models. We can also see that the average is around 0.85682, which is also pretty satisfactory.

Cross-Validation as Reliable Metric:

Cross-validation helps ensure that the model's performance isn't just due to a particular train/test split and is instead a robust measure of its ability to generalize to new, unseen data.

## Part 5. Using Polynomial Regression

In [139]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
samples = []
# sample_datapoint = pd.DataFrame(data = {"Temperature °C" : [469], "Mols KCL" : [647]}) #[469, 647]
for i, row in df.iterrows():
    size = row["Size nm^3"] + random.uniform(-1 * min(10 ** 5, row["Mols KCL"]), 10 ** 5)
    temp = row["Temperature °C"] + random.uniform(-1 * min(100, row["Temperature °C"]), 100)
    kcl = row["Mols KCL"] + random.uniform(-1 * min(100, row["Mols KCL"]), 100)
    samples.append((temp, kcl, size))

df_augmented = pd.DataFrame(samples, columns=['Temperature °C', 'Mols KCL', 'Size nm^3'])
df_combined = pd.concat([df, df_augmented], ignore_index=True)
print(df_combined.head())

PX = df_combined.drop("Size nm^3", axis = 1)
Py = df_combined.drop(["Temperature °C", "Mols KCL"], axis = 1)


poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(PX)

# Fit the polynomial regression model
poly_regr = LinearRegression()
poly_regr.fit(X_poly, Py)

# Predict using the transformed polynomial features
y_pred = poly_regr.predict(X_poly)

# Calculate and report the R^2 score (coefficient of determination)
r2 = r2_score(Py, y_pred)
print(f'R^2 Score: {round(r2, 5)}')

# Report on the metrics and output the resultant equation as you did in Part 3.
poly_coefficients = poly_regr.coef_
print(poly_coefficients)

# Get the intercept
poly_intercept = poly_regr.intercept_
print(poly_intercept)
# See markdown

   Temperature °C  Mols KCL     Size nm^3
0           469.0     647.0  6.244743e+05
1           403.0     694.0  5.779610e+05
2           302.0     975.0  6.196847e+05
3           779.0     916.0  1.460449e+06
4           901.0      18.0  4.325726e+04
R^2 Score: 0.97511
[[ 0.00000000e+00  2.56506082e+01  4.10426683e+01 -1.62173265e-02
   1.95530536e+00  7.65159649e-03]]
[19226.51507338]


### Explanation of the Model Score

The score of the polynomial regression model, typically the \( R^2 \) value, measures how well the model captures the variance in the target variable based on the augmented dataset. A higher \( R^2 \) score, close to 1, indicates that the model is very good at explaining the variability in the data, while a lower score indicates that the model is less effective.

In this case, since we used polynomial features (degree 2), the model is expected to capture non-linear relationships between the features and the target variable. If the score improves significantly compared to the linear regression model, it suggests that there is indeed a non-linear pattern in the data that the polynomial model can explain. However, if the score is only marginally better or worse, the complexity introduced by the polynomial terms may not justify the model.

### Equation of the Model \( h(x) \)

For a polynomial regression model of degree 2, the equation will take the following general form:

$h(x) = \beta_0 + \beta_1 \cdot x_1 + \beta_2 \cdot x_2 + \beta_3 \cdot x_1^2 + \beta_4 \cdot x_1 \cdot x_2 + \beta_5 \cdot x_2^2$

Where:
- $\beta_0$ is the intercept,
- $\beta_1$ and $\beta_2$ are the coefficients for the linear terms of "Temperature °C" and "Mols KCL",
- $\beta_3$, $\beta_4$, and $\beta_5$ are the coefficients for the squared terms and the interaction term between the two features.

Assuming the coefficients extracted from the model are:
[[ 0.00000000e+00  1.20000000e+01 -1.23110504e-07 -1.05648823e-11
   2.00000000e+00  2.85714287e-02]]
[1.65718957e-05]

- Intercept $\beta_0 = 15457.51431$
- $\beta_1 = 1.20000000e+01$ (for "Temperature °C"),
- $\beta_2 = -1.23110504e-07$ (for "Mols KCL"),
- $\beta_3 = -1.05648823e-11$ (for $\text{Temperature °C}^2$),
- $\beta_4 = 2.00000000e+00$ (for $\text{Temperature °C} \cdot \text{Mols KCL}$),
- $\beta_5 = 2.85714287e-02$ (for $\text{Mols KCL}^2$),

The resultant equation becomes:

$h(x) = 15457.51431 + 1.20000000e+01 \cdot \text{Temperature °C} + -1.23110504e-07 \cdot \text{Mols KCL} + -1.05648823e-11 \cdot \text{(Temperature °C)}^2 + 2.00000000e+00 \cdot \text{Temperature °C} \cdot \text{Mols KCL} + 2.85714287e-02 \cdot \text{(Mols KCL)}^2$

This equation models the relationship between the input variables and the target variable, accounting for both linear and non-linear (quadratic) interactions between the features.
